# Advanced LLM Training on Kaggle (T4 GPU)

Train a hybrid Mamba-2 + Transformer + MoE model on Kaggle T4.

**Runtime:** T4 GPU (15GB VRAM), 9-hour session

**Setup:** Upload this notebook to Kaggle, enable GPU in Settings.

## 1. Setup Environment

In [ ]:
!nvidia-smi
!python --version
!pip install bitsandbytes --quiet

In [ ]:
# Clone the project
import os
if not os.path.exists('/kaggle/working/AI'):
    !git clone https://github.com/Lamiya880/AI.git /kaggle/working/AI

os.chdir('/kaggle/working/AI')
!ls -la

## 2. Download & Tokenize Data

In [ ]:
# Download 500K samples (~100M tokens)
!python scripts/download_data.py \
    --dataset HuggingFaceFW/fineweb-edu \
    --subset sample-10BT \
    --num-samples 500000 \
    --train-tokenizer \
    --vocab-size 32000 \
    --output data/train_tokens.pt \
    --tokenizer-path tokenizer.json

## 3. Configure Training

In [ ]:
import torch
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

gpu_name = torch.cuda.get_device_name()
props = torch.cuda.get_device_properties(0)
gpu_mem = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
print(f'GPU: {gpu_name}, Memory: {gpu_mem:.1f} GB')

# Kaggle T4 config
batch_size = 1
grad_accum = 32
seq_len = 512
max_steps = 20000

print(f'\nTraining Config:')
print(f'  Batch size: {batch_size}')
print(f'  Gradient accumulation: {grad_accum}')
print(f'  Sequence length: {seq_len}')
print(f'  Max steps: {max_steps}')
print(f'  Effective batch: {batch_size * grad_accum * seq_len:,} tokens')
print(f'  Est. time: {max_steps * 1.0 / 3600:.1f} - {max_steps * 2.0 / 3600:.1f} hours')

## 4. Train the Model

In [ ]:
import sys
sys.path.insert(0, 'src')

from torch.utils.data import DataLoader
from model.config import ModelConfig
from model.transformer import HybridModel
from training.data import PreTokenizedDataset
from training.trainer import Trainer

# Model config - fits T4 15GB
model_config = ModelConfig(
    d_model=768,
    n_layers=12,
    vocab_size=32000,
    max_seq_len=512,
    attention_layer_indices=list(range(12)),  # All attention
    d_state=64,
    d_conv=4,
    expand=2,
    n_heads=12,
    n_kv_heads=4,
    head_dim=64,
    num_experts=4,
    num_shared_experts=1,
    moe_top_k=2,
    expert_dim=1024,
)

model = HybridModel(model_config)
params = model.count_parameters()
print(f'Model: {params["total"]:,} total params, {params["trainable"]:,} trainable')
print(f'Architecture: 12 attention layers + SwiGLU FFN')

In [ ]:
# Load dataset
data_path = 'data/train_tokens.pt'
if not os.path.exists(data_path):
    print(f'Data not found at {data_path}. Run the data download cell first.')
else:
    dataset = PreTokenizedDataset(data_path, seq_len)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True,
    )
    print(f'Dataset: {len(dataset):,} samples, {len(dataloader):,} batches')

In [ ]:
# Train
trainer = Trainer(
    model=model,
    train_dataloader=dataloader,
    lr=3e-4,
    weight_decay=0.1,
    warmup_steps=1000,
    stable_steps=max_steps - 2000,
    decay_steps=1000,
    grad_accum_steps=grad_accum,
    max_grad_norm=1.0,
    checkpoint_dir='checkpoints',
    log_interval=100,
    save_interval=5000,
    device='cuda',
    use_amp=True,
    optimizer_name='adamw8bit',
)

print(f'Starting training for {max_steps} steps...')
history = trainer.train(max_steps=max_steps)
print(f'Training complete! Final loss: {history["train_loss"][-1]:.4f}')

## 5. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['train_loss'])
ax1.set_xlabel('Log Step')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss (Kaggle T4)')
ax1.grid(True)

ax2.plot(history['lr'])
ax2.set_xlabel('Log Step')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule (WSD)')
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150)
plt.show()

## 6. Generate Text

In [ ]:
from inference.generate import generate
from training.tokenizer import load_tokenizer
from training.checkpoint import load_checkpoint

# Load best model
if os.path.exists('checkpoints/best_model.pt'):
    ckpt = torch.load('checkpoints/best_model.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model'])
    print('Loaded best model checkpoint')

model.eval()
tokenizer = load_tokenizer('tokenizer.json')

prompts = [
    "The future of artificial intelligence",
    "Once upon a time in a land far away",
    "The scientific method involves",
]

for prompt in prompts:
    input_ids = torch.tensor([tokenizer.encode(prompt).ids], device='cuda')
    output_ids = generate(model, input_ids, max_new_tokens=150, temperature=0.8, top_p=0.9)
    text = tokenizer.decode(output_ids[0].tolist())
    print(f'\n--- Prompt: {prompt} ---')
    print(text)

## 7. Export Model

In [ ]:
from inference.export import export_safetensors

export_path = export_safetensors(model.cpu(), 'export')
print(f'Exported to: {export_path}')
!ls -lh export/

In [ ]:
# Create output for Kaggle
import shutil
shutil.make_archive('/kaggle/working/model_export', 'zip', 'export')
print('Model exported to /kaggle/working/model_export.zip')
print('Download from Kaggle Output panel on the right →')